# 🤖 AI Resume Screening System
### Built with LangChain + LangSmith Tracing

---
**What this does:**  
Takes a resume and a job description, then figures out how well the candidate fits —  
giving a score from 0 to 100, plus a plain-English explanation of *why*.

**Pipeline:**  
`Resume → Skill Extraction → Matching → Scoring → Explanation → LangSmith Trace`

## Step 0 — Install Dependencies

In [ ]:
# Run this cell once to install everything you need
# After installing, restart your kernel before moving on
!pip install langchain langchain-openai langsmith python-dotenv -q

## Step 1 — Set Up API Keys & Enable Tracing

Create a `.env` file in the same folder with these values:
```
OPENAI_API_KEY=sk-...
LANGCHAIN_API_KEY=ls__...
LANGCHAIN_PROJECT=resume-screener
```
Get your LangSmith key at: https://smith.langchain.com

In [ ]:
import os
from dotenv import load_dotenv

# Load keys from .env so we don't hardcode anything
load_dotenv()

# This single line is all it takes to turn on LangSmith tracing.
# Every .invoke() call after this will show up in your LangSmith dashboard.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = os.getenv("LANGCHAIN_PROJECT", "resume-screener")

print("✅ Tracing is ON — check https://smith.langchain.com for your runs")
print(f"   Project: {os.environ['LANGCHAIN_PROJECT']}")

## Step 2 — Load the Language Model

In [ ]:
from langchain_openai import ChatOpenAI

# Using gpt-4o-mini — smart enough for this task and easy on the wallet
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,        # zero temp = consistent, deterministic output (good for scoring)
    api_key=os.getenv("OPENAI_API_KEY")
)

print("✅ Model ready:", llm.model_name)

## Step 3 — Define Our Three Prompts

We split the work into three focused prompts instead of one big messy one.  
Each prompt has one job — that's what makes this modular and easy to debug.

In [ ]:
from langchain.prompts import PromptTemplate

# ── Prompt 1: Skill Extraction ─────────────────────────────────────────────
# Pull out the candidate's skills, tools, and years of experience.
# The strict "DO NOT" rules prevent the model from inventing things.

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are a technical recruiter reading a resume.

Extract ONLY what is explicitly written in the resume below.
DO NOT infer or assume skills that are not clearly mentioned.
DO NOT add generic skills like 'communication' unless stated.

Resume:
{resume}

Return your answer in this exact format:

SKILLS: <comma-separated list of technical skills>
TOOLS: <comma-separated list of tools/frameworks/libraries>
EXPERIENCE: <total years of relevant experience as a number>
EDUCATION: <highest degree and field>
PROJECTS: <one-line summary of notable projects, or 'None mentioned'>
"""
)


# ── Prompt 2: Match + Score ────────────────────────────────────────────────
# Compare what the candidate has vs what the job needs, then give a 0–100 score.
# We feed it the extracted profile so this step never sees the raw resume —
# that's what keeps hallucination risk low.

scoring_prompt = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
You are an unbiased hiring evaluator. Score the candidate strictly based on facts.

CANDIDATE PROFILE (extracted from resume):
{extracted_profile}

JOB DESCRIPTION:
{job_description}

Scoring rules:
- Compare skills/tools mentioned in the profile against requirements in the JD
- Award points only for matches that are explicitly present
- Do NOT give benefit of the doubt for missing information
- Score range: 0 (no match) to 100 (perfect match)

Respond in this format:

MATCHED_SKILLS: <skills from profile that appear in JD requirements>
MISSING_SKILLS: <skills required by JD but absent from profile>
EXPERIENCE_FIT: <Yes/No/Partial — with one sentence why>
FIT_SCORE: <integer 0–100>
"""
)


# ── Prompt 3: Explanation ──────────────────────────────────────────────────
# Turn the score into something a real recruiter can read and act on.
# We include the score so the model explains *that specific number*, not a generic answer.

explanation_prompt = PromptTemplate(
    input_variables=["scoring_result", "fit_score"],
    template="""
You are writing a hiring recommendation for a recruiter who isn't technical.
Be direct, honest, and use plain language. No fluff.

Scoring Result:
{scoring_result}

Fit Score: {fit_score} / 100

Write a short recommendation (3–5 sentences) that covers:
1. Whether to proceed with this candidate and why
2. Their strongest relevant strengths
3. The biggest gap(s) that might be a concern
4. A clear hiring suggestion (Strongly Recommend / Recommend / Maybe / Do Not Recommend)

End with: VERDICT: <one of the four options above>
"""
)

print("✅ All three prompts defined")

## Step 4 — Build the Pipeline with LCEL

LCEL (LangChain Expression Language) lets us chain steps with `|` (pipe operator).  
Each step's output flows automatically into the next.

In [ ]:
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough
import re

# Simple string parser — strips the ChatMessage wrapper and gives us plain text
parser = StrOutputParser()


# ── Individual chains (one per prompt) ─────────────────────────────────────

extract_chain = extraction_prompt | llm | parser
score_chain   = scoring_prompt   | llm | parser
explain_chain = explanation_prompt | llm | parser


# ── Helper: pull the numeric score out of the scoring step's text output ───
def parse_score(scoring_text: str) -> str:
    """Grabs the integer after 'FIT_SCORE:' so we can pass it to the explain step."""
    match = re.search(r"FIT_SCORE:\s*(\d+)", scoring_text)
    return match.group(1) if match else "N/A"


# ── Full pipeline — this is the main thing we call ─────────────────────────
# Input:  {"resume": "...", "job_description": "..."}
# Output: {"extracted": ..., "scored": ..., "score": ..., "explanation": ...}

def run_screening(resume: str, job_description: str) -> dict:
    """
    Runs the full 3-step screening pipeline:
      1. Extract skills/experience from the resume
      2. Match against the job description and score it
      3. Generate a plain-English explanation and hiring verdict

    All three steps are traced automatically in LangSmith.
    """
    # Step 1: What does this candidate actually have?
    extracted = extract_chain.invoke({"resume": resume})

    # Step 2: How well does it match the job?
    scored = score_chain.invoke({
        "extracted_profile": extracted,
        "job_description": job_description
    })

    # Step 3: What should the recruiter actually do with this info?
    score_value = parse_score(scored)
    explanation = explain_chain.invoke({
        "scoring_result": scored,
        "fit_score": score_value
    })

    return {
        "extracted": extracted,
        "scored": scored,
        "score": score_value,
        "explanation": explanation
    }


print("✅ Pipeline ready — call run_screening(resume, job_description) to screen a candidate")

## Step 5 — Sample Data: 3 Candidates + 1 Job Description

Three realistic candidates — strong, average, and weak — against a Data Scientist JD.

In [ ]:
# ── Job Description ─────────────────────────────────────────────────────────

job_description = """
Position: Data Scientist
Company: DataFlow Analytics

Requirements:
- 3+ years of experience in data science or machine learning
- Strong Python skills (pandas, numpy, scikit-learn)
- Experience with deep learning frameworks (TensorFlow or PyTorch)
- Proficiency in SQL and data wrangling
- Familiarity with cloud platforms (AWS, GCP, or Azure)
- Experience deploying ML models to production
- Good communication — ability to explain findings to non-technical stakeholders
- Master's or PhD in CS, Statistics, or related field preferred
"""


# ── Candidate 1: Strong ─────────────────────────────────────────────────────
# Has almost everything the job asks for

resume_strong = """
Priya Sharma — Data Scientist
Email: priya@email.com | LinkedIn: linkedin.com/in/priyasharma

EXPERIENCE
Senior Data Scientist — TechCorp (2021–2024, 3 years)
  - Built customer churn prediction model (PyTorch) with 91% accuracy, deployed on AWS SageMaker
  - Reduced model inference time by 40% using ONNX optimization
  - Presented monthly insights to C-suite, simplifying complex ML results into business language

Data Analyst — InfoSys (2019–2021, 2 years)
  - Created automated ETL pipelines using Python (pandas, numpy) and PostgreSQL
  - Managed 50M+ row datasets on GCP BigQuery

EDUCATION
M.Sc. in Data Science — IIT Bombay (2019)

SKILLS
Python, pandas, numpy, scikit-learn, PyTorch, TensorFlow (basic), SQL, PostgreSQL

TOOLS
AWS SageMaker, GCP BigQuery, Docker, Git, Jupyter, MLflow

PROJECTS
NLP Sentiment Analyzer — fine-tuned BERT model for product review classification (PyTorch, HuggingFace)
"""


# ── Candidate 2: Average ────────────────────────────────────────────────────
# Some experience, missing a few key requirements

resume_average = """
Rahul Nair — Junior Data Scientist
Email: rahul@email.com

EXPERIENCE
Data Analyst — StartupXYZ (2022–2024, 2 years)
  - Built dashboards using Python and matplotlib to track product KPIs
  - Wrote SQL queries for data extraction and reporting
  - Trained basic classification models with scikit-learn

EDUCATION
B.Tech in Computer Science — VIT University (2022)

SKILLS
Python, pandas, scikit-learn, SQL, data visualization

TOOLS
MySQL, Excel, Tableau, Jupyter Notebook, Git

PROJECTS
Sales Forecasting using Linear Regression (scikit-learn) — personal project
"""


# ── Candidate 3: Weak ───────────────────────────────────────────────────────
# Has some Python and SQL, but no real data science background

resume_weak = """
Amit Verma — Software Developer
Email: amit@email.com

EXPERIENCE
Web Developer — Freelance (2021–2024, 3 years)
  - Built e-commerce websites using Django and React
  - Wrote REST APIs in Python (Flask)
  - Used MySQL for database management

EDUCATION
B.Sc. in Information Technology — Mumbai University (2021)

SKILLS
Python, JavaScript, HTML/CSS, Django, Flask, REST APIs

TOOLS
MySQL, Git, Postman, VS Code, Linux

PROJECTS
Online food delivery platform — full-stack web application
"""


candidates = [
    {"name": "Priya Sharma (Strong)",  "resume": resume_strong},
    {"name": "Rahul Nair (Average)",   "resume": resume_average},
    {"name": "Amit Verma (Weak)",      "resume": resume_weak},
]

print("✅ 3 candidates and 1 job description ready")

## Step 6 — Run the Screening Pipeline

This is where the actual LLM calls happen. Each candidate generates 3 separate LangSmith traces.

In [ ]:
results = {}

for candidate in candidates:
    name = candidate["name"]
    print(f"\n{'='*60}")
    print(f"🔍 Screening: {name}")
    print(f"{'='*60}")

    result = run_screening(candidate["resume"], job_description)
    results[name] = result

    print(f"\n📋 EXTRACTED PROFILE:")
    print(result["extracted"])

    print(f"\n📊 SCORING BREAKDOWN:")
    print(result["scored"])

    print(f"\n💬 RECRUITER RECOMMENDATION:")
    print(result["explanation"])

    print(f"\n✅ Final Score: {result['score']} / 100")

## Step 7 — Summary Comparison Table

In [ ]:
import re

def extract_verdict(explanation_text: str) -> str:
    """Pull the VERDICT line from the explanation output."""
    match = re.search(r"VERDICT:\s*(.+)", explanation_text)
    return match.group(1).strip() if match else "—"


print("\n" + "═"*65)
print("  CANDIDATE SCREENING SUMMARY")
print("═"*65)
print(f"{'Candidate':<30} {'Score':>7}   {'Verdict'}")
print("-"*65)

for name, result in results.items():
    verdict = extract_verdict(result["explanation"])
    print(f"{name:<30} {result['score']:>5}/100   {verdict}")

print("═"*65)

## Step 8 — Intentional Bug Demo (for LangSmith Debugging)

The assignment asks us to show at least **one incorrect output** being caught in LangSmith.  
Here we intentionally send a malformed prompt to trigger a bad response, then fix it.

In [ ]:
# ── Buggy version — missing the 'resume' key in the input ──────────────────
# This causes the model to get an empty/garbage resume and return nonsense.
# In LangSmith, this run will show up with odd output — easy to spot and fix.

print("🐛 Running intentionally broken version for LangSmith debugging demo...\n")

try:
    # Bug: passing job_description content into the 'resume' field by mistake
    buggy_result = extract_chain.invoke({"resume": ""})
    print("Output from buggy run (should look wrong/empty):")
    print(buggy_result)
except Exception as e:
    print(f"Error: {e}")

print("\n" + "-"*50)
print("✅ Fixed version (correct input):")
print("-"*50)

# Fixed: using the actual resume string
fixed_result = extract_chain.invoke({"resume": resume_strong})
print(fixed_result)

print("\n💡 Both runs are now visible in LangSmith — compare the buggy vs. fixed traces.")

## Step 9 — Bonus: JSON Output + Few-Shot Prompting

Optional but recommended. Returns structured JSON instead of raw text —  
much easier to use if you're building an app on top of this.

In [ ]:
import json

json_scoring_prompt = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
You are a hiring evaluator. Score the candidate and return ONLY valid JSON.
Do not include any text before or after the JSON object.

CANDIDATE PROFILE:
{extracted_profile}

JOB DESCRIPTION:
{job_description}

Few-shot examples of the expected format:

Example 1 (strong match):
{{"fit_score": 88, "matched_skills": ["Python", "PyTorch", "AWS"], "missing_skills": ["Azure"], "verdict": "Strongly Recommend", "summary": "Excellent technical fit with proven production ML experience."}}

Example 2 (weak match):
{{"fit_score": 22, "matched_skills": ["Python"], "missing_skills": ["PyTorch", "TensorFlow", "SQL", "Cloud"], "verdict": "Do Not Recommend", "summary": "Background is in web development, not data science."}}

Now evaluate the actual candidate above and return JSON in the exact same format:
"""
)

json_chain = json_scoring_prompt | llm | parser


def screen_with_json(resume: str, job_desc: str) -> dict:
    """Run screening and return structured JSON output instead of raw text."""
    extracted = extract_chain.invoke({"resume": resume})
    raw_json  = json_chain.invoke({"extracted_profile": extracted, "job_description": job_desc})

    # Clean up any accidental markdown fences before parsing
    clean_json = raw_json.strip().removeprefix("```json").removesuffix("```").strip()

    try:
        return json.loads(clean_json)
    except json.JSONDecodeError:
        # If parsing fails, return the raw string so you can debug it
        return {"error": "JSON parse failed", "raw": raw_json}


print("Running bonus JSON screening on all 3 candidates...\n")

for candidate in candidates:
    output = screen_with_json(candidate["resume"], job_description)
    print(f"\n{'—'*50}")
    print(f"Candidate: {candidate['name']}")
    print(json.dumps(output, indent=2))

---

## 📌 LangSmith Checklist

After running all cells, go to https://smith.langchain.com and confirm:

- [ ] Project **resume-screener** appears in the left sidebar
- [ ] At least **3 runs** visible (one per candidate)
- [ ] Each run shows **3 steps**: extraction → scoring → explanation
- [ ] The **buggy run** (empty resume) is also visible with a different output
- [ ] You can click into any step and see the exact prompt + response

Take screenshots of these for your submission.

---

## 📂 Project Structure

```
resume_screener/
├── AI_Resume_Screener.ipynb   ← this notebook (all-in-one)
├── prompts/
│   └── prompts.py             ← standalone prompt definitions
├── chains/
│   └── pipeline.py            ← standalone chain definitions
├── main.py                    ← script version (run from terminal)
├── .env                       ← your API keys (don't commit this!)
└── requirements.txt
```

---

*Built for: Data Science Internship Assignment — February 2026*